# 21 · Matryoshka representations and full-vector rescoring

## The engineering question
Can trained embedding prefixes preserve useful retrieval while reducing candidate-search width?

**Learning outcomes:** distinguish trained nested representations from arbitrary slicing; renormalize each prefix; measure semantic quality and full-vector candidate recall; rescore a bounded candidate pool; account for both stored representations.

Bootstrap the separate Apache-licensed model with `uv run vmc bootstrap --model matryoshka`. The pinned mxbai-embed-large-v1 produces 1,024 dimensions and supports Matryoshka truncation. Query text receives the model's documented retrieval prefix; documents do not. We compare 128, 256, 512, and 1,024 dimensions, fixed before inspecting these development results. Full-vector rescoring uses five candidates and returns three documents.

Matryoshka training applies objectives at nested widths so early coordinates carry useful information. Merely slicing a conventional encoder does not supply that training. The MiniLM control below illustrates arbitrary truncation; it has different weights, size, and training data, so this is not a causal ablation isolating Matryoshka training.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("21", "B", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import time
from uuid import uuid4
from qdrant_client import QdrantClient, models as qm
from vector_masterclass.models import embed, release_models
from vector_masterclass.data import documents, queries
from vector_masterclass.retrieval import exact_search, normalized
from vector_masterclass.metrics import ranking_metrics, ann_recall
from vector_masterclass.config import qdrant_url, model_config, fingerprint

docs = documents(tenant="public")
qs = queries("development")
ids = [d.key for d in docs]
rows, costs = [], []
matrices = {}
for key, widths in [("matryoshka", [128, 256, 512, 1024]), ("dense", [128, 256, 384])]:
    started = time.perf_counter()
    full = embed([d.text for d in docs], key=key, batch_size=4)
    queries_full = embed([q.text for q in qs], key=key, role="query", batch_size=4)
    encode_seconds = time.perf_counter() - started
    matrices[key] = (full, queries_full)
    for width in widths:
        short = normalized(full[:, :width])
        qshort = normalized(queries_full[:, :width])
        costs.append(
            {
                "model": key,
                "dimensions": width,
                "short_vectors_bytes": short.nbytes,
                "full_plus_short_bytes": full.nbytes + short.nbytes,
                "shared_full_encoding_seconds": encode_seconds,
            }
        )
        for q, qf, ql in zip(qs, queries_full, qshort):
            started = time.perf_counter()
            candidates = [i for i, s in exact_search(ql, short, ids, k=5)]
            oracle = [i for i, s in exact_search(qf, full, ids, k=3)]
            direct = candidates[:3]
            rescored = sorted(candidates, key=lambda id_: (-float(full[ids.index(id_)] @ qf), id_))[
                :3
            ]
            elapsed = time.perf_counter() - started
            for mode, ranked in [
                ("prefix", direct),
                ("prefix candidates + full rescore", rescored),
            ]:
                rows.append(
                    {
                        "model": key,
                        "dimensions": width,
                        "query": q.id,
                        "mode": mode,
                        "full_vector_top3_candidate_recall": len(set(oracle) & set(candidates)) / 3,
                        "local_scoring_seconds": elapsed,
                        **ranking_metrics(ranked, q.relevance, 3),
                    }
                )
    release_models()
# Real named-vector candidate retrieval and full-width rescoring in Qdrant.
full, qfull = matrices["matryoshka"]
short = normalized(full[:, :256])
client = QdrantClient(url=qdrant_url(), timeout=30)
name = "vmc_21_" + uuid4().hex[:8]
client.create_collection(
    name,
    vectors_config={
        str(d): qm.VectorParams(size=d, distance=qm.Distance.COSINE) for d in [256, 1024]
    },
    metadata={"model_signature": fingerprint(model_config("matryoshka")), "prefix_dimensions": 256},
)
try:
    client.upsert(
        name,
        points=[
            qm.PointStruct(id=i, vector={"256": short[i].tolist(), "1024": full[i].tolist()})
            for i in range(len(docs))
        ],
        wait=True,
    )
    served = client.query_points(
        name,
        prefetch=qm.Prefetch(
            query=normalized(qfull[0, :256]).tolist(),
            using="256",
            limit=5,
            params=qm.SearchParams(exact=True),
        ),
        query=qfull[0].tolist(),
        using="1024",
        limit=3,
        search_params=qm.SearchParams(exact=True),
    ).points
    expected_candidates = [i for i, s in exact_search(normalized(qfull[0, :256]), short, ids, k=5)]
    expected = sorted(
        expected_candidates, key=lambda id_: (-float(full[ids.index(id_)] @ qfull[0]), id_)
    )[:3]
    served_ids = [ids[h.id] for h in served]
finally:
    client.delete_collection(name)
    client.close()
results = lesson.table(pd.DataFrame(rows))
costs = lesson.table(pd.DataFrame(costs), "representation-cost")
display(
    results.groupby(["model", "dimensions", "mode"])[
        ["recall", "ndcg", "full_vector_top3_candidate_recall"]
    ].mean()
)
display(costs)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
summary = results.groupby(["model", "dimensions", "mode"]).ndcg.mean().reset_index()
for (key, mode), group in summary.groupby(["model", "mode"]):
    axes[0].plot(group.dimensions, group.ndcg, "o-", label=key + " / " + mode)
axes[0].set(
    xlabel="Prefix dimensions",
    ylabel="Mean development nDCG@3",
    ylim=(0, 1.05),
    title="Nested width is a quality/storage tradeoff",
)
axes[0].legend(fontsize=6, loc="lower right")
part = costs[costs.model == "matryoshka"]
axes[1].plot(part.dimensions, part.short_vectors_bytes, "o-", label="prefix only")
axes[1].plot(part.dimensions, part.full_plus_short_bytes, "o-", label="full + prefix")
axes[1].set(
    xlabel="Prefix dimensions",
    ylabel="Raw vector bytes",
    title="Rescoring retains original-vector storage",
)
axes[1].legend()
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
Prefix encoding still runs the full transformer in this implementation. It reduces vector storage/search arithmetic, not transformer inference cost. Full-vector rescoring repairs order only within candidates. A small corpus may show no losses at any tested width; that does not establish a safe width for a million-document corpus or another language/domain.

**Exercise:** lower the candidate budget from five to three on development data. Find a missing full-width neighbor that rescoring cannot recover. Estimate two-representation storage and compare with a single full-width collection. Explain why changing width requires a new collection/vector schema and a versioned query contract.

Contrastive training needs meaningful positives and negatives. Hard negatives can teach fine distinctions but can also contain unlabeled positives; distillation transfers a teacher's ranking and its biases. This notebook performs inference with trained weights, not frontier model training or a reproduction of the model card's benchmark results. [Pinned model card](https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1), [Matryoshka representation learning](https://arxiv.org/abs/2205.13147).

In [ ]:
manifest = lesson.finish(
    {
        "trained_full_width": matrices["matryoshka"][0].shape[1] == 1024,
        "seven_width_configurations": len(costs) == 7,
        "real_server_rescore_matches": served_ids == expected,
        "candidate_recall_bounded": bool(
            results.full_vector_top3_candidate_recall.between(0, 1).all()
        ),
        "ordinary_truncation_control": set(results.model) == {"dense", "matryoshka"},
    },
    observations={
        "split": "development",
        "candidate_budget": 5,
        "final_k": 3,
        "query_prompt": model_config("matryoshka")["query_prefix"],
        "encoding_reused_across_widths": True,
        "Qdrant_experiment": "exact tiny named-vector prefetch/rescore; no ANN latency claim",
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.